# 🔴🔵 Cannon & Wall — GRPO Training

**Model:** Qwen/Qwen2.5-3B-Instruct (4-bit LoRA) | **Steps:** 50

Each step: generate 4 completions → score via live env → GRPO update.

> **Runtime:** Go to Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Cell 1 — Install
!pip install -q transformers accelerate peft bitsandbytes wandb httpx matplotlib

In [ ]:
# Cell 2 — Imports + W&B + Load Model
import os, json, re, random, torch, httpx, wandb
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import get_peft_model, LoraConfig

wandb.init(project="cannon-and-wall", name="grpo-training-run",
           config={"model": "Qwen/Qwen2.5-3B-Instruct", "algorithm": "GRPO", "max_steps": 50})
print("W&B run:", wandb.run.url)

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 2048

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config,
                                              device_map="auto", torch_dtype=torch.float16)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
model.print_trainable_parameters()
print("\u2705 Model loaded")

In [ ]:
# Cell 3 — Environment client + helpers
ENV_URL = "https://cystroncode-cannon-and-wall.hf.space"

def env_reset(stage=1):
    r = httpx.post(f"{ENV_URL}/reset", params={"stage": stage}, timeout=30)
    r.raise_for_status()
    return r.json()

def env_step(action):
    r = httpx.post(f"{ENV_URL}/step", json=action, timeout=30)
    r.raise_for_status()
    return r.json()

CANNON_SYSTEM = (
    "You are a security researcher (Cannon). "
    "Analyse the Flask app source below and identify ONE vulnerability. "
    "Respond ONLY with valid JSON: "
    '{"agent":"cannon","vuln_type":"sqli|xss|broken_auth",'
    '"line_number":<int>,"explanation":"...","proof_of_concept":"..."}')

def build_prompt(source_code):
    return f"{CANNON_SYSTEM}\n\n### Source Code\n```python\n{source_code}\n```"

def generate(prompt, max_tokens=256):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LEN).to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7,
                             do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def score_completion(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        action = json.loads(match.group()) if match else {}
        action["agent"] = "cannon"
        env_reset(stage=1)
        result = env_step(action)
        return float(result.get("reward", 0.0)), result.get("breakdown", {})
    except Exception:
        return 0.0, {}

# Quick test
obs = env_reset(stage=1)
print("\u2705 Env connected. Source code length:", len(obs.get("source_code", "")))

In [ ]:
# Cell 4 — GRPO Training (50 gradient steps)
from torch.optim import AdamW

G = 4            # group size
NUM_STEPS = 50
LR = 5e-5

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
obs = env_reset(stage=1)
prompt = build_prompt(obs["source_code"])
prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN)["input_ids"]
prompt_len = prompt_ids.shape[1]

rewards_history = []
print(f"Starting GRPO — {NUM_STEPS} steps, G={G}, prompt={prompt_len} tokens\n")

for step in range(NUM_STEPS):
    # 1) Generate G completions
    completions = [generate(prompt) for _ in range(G)]

    # 2) Score each via live environment
    step_rewards = []
    for c in completions:
        r, bd = score_completion(c)
        step_rewards.append(r)

    mean_r = sum(step_rewards) / G
    advantages = [r - mean_r for r in step_rewards]
    rewards_history.append(step_rewards)

    # 3) Policy gradient update
    model.train()
    optimizer.zero_grad()
    valid = 0
    for comp, adv in zip(completions, advantages):
        if abs(adv) < 1e-8:
            continue
        full = prompt + comp
        enc = tokenizer(full, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)
        labels = enc["input_ids"].clone()
        labels[0, :prompt_len] = -100  # mask prompt tokens
        out = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], labels=labels)
        loss = adv * out.loss / G
        loss.backward()
        valid += 1

    if valid > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    wandb.log({"step": step, "reward/mean": mean_r,
               "reward/max": max(step_rewards), "reward/min": min(step_rewards)})
    if step % 5 == 0:
        print(f"Step {step:3d} | Reward mean={mean_r:.3f}  max={max(step_rewards):.3f}")

print("\n\u2705 Training complete. Model weights updated via gradient descent.")

In [ ]:
# Cell 5 — Evaluation + Reward Curve + Save
import matplotlib.pyplot as plt
import numpy as np

# --- Baseline vs Trained ---
RANDOM_ACTIONS = [
    {"agent":"cannon","vuln_type":"sqli","line_number":5,"explanation":"r","proof_of_concept":"' OR 1=1--"},
    {"agent":"cannon","vuln_type":"xss","line_number":10,"explanation":"r","proof_of_concept":"<script>alert(1)</script>"},
    {"agent":"cannon","vuln_type":"broken_auth","line_number":15,"explanation":"r","proof_of_concept":"?user=admin"},
]

def random_baseline(n=10):
    total = 0.0
    for _ in range(n):
        a = random.choice(RANDOM_ACTIONS).copy()
        try:
            env_reset(1); total += float(env_step(a).get("reward", 0.0))
        except: pass
    return total / n

def eval_trained(n=10):
    total = 0.0
    p = build_prompt(env_reset(1)["source_code"])
    for _ in range(n):
        raw = generate(p)
        r, _ = score_completion(raw)
        total += r
    return total / n

baseline = random_baseline()
trained = eval_trained()
print(f"Baseline reward : {baseline:.3f}")
print(f"Trained reward  : {trained:.3f}")
print(f"Improvement     : {trained - baseline:+.3f}")

wandb.log({"eval/baseline": baseline, "eval/trained": trained, "eval/improvement": trained - baseline})

# --- Reward Curve ---
means = [np.mean(r) for r in rewards_history]
rolling = [np.mean(means[max(0,i-9):i+1]) for i in range(len(means))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(means, color="#E85D4A", alpha=0.3, label="Per-step mean")
ax.plot(rolling, color="#B33025", linewidth=2.5, label="10-step rolling avg")
ax.axhline(y=baseline, color="gray", linestyle="--", alpha=0.5, label=f"Random baseline ({baseline:.2f})")
ax.set_title("Cannon & Wall \u2014 GRPO Training | Qwen2.5-3B | 50 Steps", fontweight="bold")
ax.set_xlabel("Training Step"); ax.set_ylabel("Reward")
ax.legend(); ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=150)
plt.show()
print("\nSaved reward_curve.png")

# --- Save Adapter ---
ADAPTER_DIR = "./cannon_grpo_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")
print("Push: model.push_to_hub('CystronCode/cannon-and-wall-grpo')")
wandb.finish()